# Apa ini?
tujuan notebook ini buat ngeliat informasi apa aja yang bisa diambil dari setiap dataset SFT yang udah jadi.

misal untuk dataset Ujaran Kebencian, kebanyakan bahas suku atau agama atau apa
untuk fitnah siapa yang paling sering diserang? pemerintah? polisi? 
dll

In [1]:
import pandas as pd
from pathlib import Path

# EDA Fitnah

In [2]:
folderData = Path("Dataset_Fitnah")
folderSFT = Path("DatasetAkhir")
hasil = folderSFT / "fitnah.jsonl"
files = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    files.append(df_sementara)

df_fitnah = pd.concat(files, ignore_index=True)

df_fitnahSFT = pd.read_json(hasil, lines=True)

In [4]:
df_fitnah.head(2)

,url,title,published_at,text,sumber_berita,label,is_layak,claim_fitnah,cot_bantah_fitnah,reasoning_fitnah,cot_bantuh_fitnah
0,https://news.indozone.id/hukum/2486475324/bnn-...,"BNN Gerebek Narkoba di Berlan Matraman, Puluha...",2025-11-25 22:05:10,"BNN menggedebek kampung narkoba di Berlan, Mat...",indozone,Kriminalitas,1.0,BNN sengaja pilih kampung miskin buat dijadika...,"[BNN melakukan penggerebekan di Berlan, Matram...",Fakta dipelintir dengan menuduh BNN memilih ka...,NaN
1,https://www.cnbcindonesia.com/mymoney/20260226...,"Video: 2026, Asuransi Perkuat Modal-Tingkatkan...",N/A,CNBC Indonesia menyelenggarakan Insurance Foru...,cnbcindonesia,Ekonomi & Bisnis,0.0,NaN,[NaN],NaN,NaN


In [5]:
df_fitnahSFT.head(2)

,instruction,input,output
0,Bandingkan narasi berikut dengan rujukan resmi...,input: BNN sengaja pilih kampung miskin buat d...,Label: **Fitnah.** penjelasan: Fakta dipelinti...
1,Tentukan kategori teks ini berdasarkan bukti-b...,input: DPRD Berau sengaja lambatkan serapan an...,Label: **Fitnah.** penjelasan: Claim ini mengg...


In [6]:
df_fitnahSFT["claim"] = df_fitnahSFT["input"].str.extract(r"input: (.*?);\n\n")[0]

In [7]:
df_fitnahSFT.head(2)

,instruction,input,output,claim
0,Bandingkan narasi berikut dengan rujukan resmi...,input: BNN sengaja pilih kampung miskin buat d...,Label: **Fitnah.** penjelasan: Fakta dipelinti...,BNN sengaja pilih kampung miskin buat dijadika...
1,Tentukan kategori teks ini berdasarkan bukti-b...,input: DPRD Berau sengaja lambatkan serapan an...,Label: **Fitnah.** penjelasan: Claim ini mengg...,DPRD Berau sengaja lambatkan serapan anggaran ...


In [8]:
df_fitnahSFT = (
    df_fitnahSFT.merge(
        df_fitnah[["claim_fitnah", "label", "published_at"]],
        left_on=["claim"],
        right_on=["claim_fitnah"],
        how="left"
    )
)

In [9]:
df_fitnahSFT.head(2)

,instruction,input,output,claim,claim_fitnah,label,published_at
0,Bandingkan narasi berikut dengan rujukan resmi...,input: BNN sengaja pilih kampung miskin buat d...,Label: **Fitnah.** penjelasan: Fakta dipelinti...,BNN sengaja pilih kampung miskin buat dijadika...,BNN sengaja pilih kampung miskin buat dijadika...,Kriminalitas,2025-11-25 22:05:10
1,Tentukan kategori teks ini berdasarkan bukti-b...,input: DPRD Berau sengaja lambatkan serapan an...,Label: **Fitnah.** penjelasan: Claim ini mengg...,DPRD Berau sengaja lambatkan serapan anggaran ...,DPRD Berau sengaja lambatkan serapan anggaran ...,Politik,2025-07-29 19:47:06


In [10]:
df_fitnahSFT["label"].unique()

array(['Kriminalitas', 'Politik', 'Bencana & Keamanan', 'Kesehatan',
       'Ekonomi & Bisnis'], dtype=object)

In [11]:
df_fitnahSFT["label"].value_counts()

label
Politik               3538
Kriminalitas          1722
Ekonomi & Bisnis       930
Bencana & Keamanan     802
Kesehatan              220
Name: count, dtype: int64

# EDA Disinformasi

In [20]:
folderData = Path("Dataset_disinformasi")
folderSFT = Path("DatasetAkhir")
hasil = folderSFT / "disinformasi.jsonl"
files = []


for file in folderData.glob("*.jsonl"):
    df_sementara = pd.read_json(file, lines=True)
    files.append(df_sementara)

df_disinformasi = pd.concat(files, ignore_index=True)

df_disinformasiSFT = pd.read_json(hasil, lines=True)

In [17]:
df_disinformasiSFT.head(2)

,instruction,input,output
0,Bandingkan pernyataan netizen ini dengan rujuk...,Klaim: Heboh! Pemuda Bakar Mobil Mercedes Kare...,Label: **Disinformasi.** penjelasan: Ini adala...
1,"Tandai klaim ini sebagai Disinformasi, Fitnah,...",Klaim: Waspada! Paket Narkoba Dikirim ke Rumah...,Label: **Disinformasi.** penjelasan: Ini adala...


In [15]:
df_disinformasi.head(2)

,claim_disinformasi,berita_asli,label,is_layak,cot_bantah_disinformasi,reasoning_disinformasi,cot_bantuh_disinformasi
0,Heboh! Pemuda Bakar Mobil Mercedes Karena Buat...,Beredar di media sosial sebuah video yang dikl...,Disinformasi,1,[Mobil yang dibakar adalah Mercedes AMG GT 63 ...,Ini adalah konten menyesatkan yang mencatut na...,NaN
1,"Waspada! Paket Narkoba Dikirim ke Rumah Warga,...",Telah beredar pesan berantai whatsaap tentang ...,Disinformasi,1,[Pesan berantai tersebut telah beredar sejak F...,Ini adalah pesan berantai yang dirancang untuk...,NaN


In [21]:
df_disinformasiSFT["claim"] = df_disinformasiSFT["input"].str.extract(r"Klaim: (.*?)\n\n")

In [23]:
df_disinformasiSFT.head(2)

,instruction,input,output,claim
0,Bandingkan pernyataan netizen ini dengan rujuk...,Klaim: Heboh! Pemuda Bakar Mobil Mercedes Kare...,Label: **Disinformasi.** penjelasan: Ini adala...,Heboh! Pemuda Bakar Mobil Mercedes Karena Buat...
1,"Tandai klaim ini sebagai Disinformasi, Fitnah,...",Klaim: Waspada! Paket Narkoba Dikirim ke Rumah...,Label: **Disinformasi.** penjelasan: Ini adala...,"Waspada! Paket Narkoba Dikirim ke Rumah Warga,..."


In [ ]:
tes = df_disinformasiSFT.merge(
    df_disinformasi[[""]]
)